In [ ]:
!pip install jupyterlab


In [ ]:
from google import genai
from google.colab import userdata


In [ ]:
api_key = userdata.get('GOOGLE_API_KEY')

In [ ]:
client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model ="gemini-2.5-flash",
    contents = "Hi",
    config = {'max_output_tokens':50}
    )
print(response.text)


In [ ]:
###reading the file

with open('/content/Food_Recipes.txt') as f:
  text = f.read()
  print(text)

In [ ]:
### create chunks of this text
### we should not create embeddings we should not pass all text
chunks = text.split('\n\n')
print(len(chunks))
chunks

In [ ]:
print(chunks[:2])

In [ ]:
###create embedding model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


In [ ]:
import numpy as np
embeddings = model.encode(chunks)
embeddings = np.array(embeddings)
print(embeddings.shape)


In [ ]:
print(embeddings[0])

In [ ]:
###create faiss - to store embeddings in Vector DataBase
!pip install faiss-cpu


In [ ]:
import faiss

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Vector Stored-", index.ntotal)

In [ ]:
## Text data- Read - Create Chunks - created embeddings - stored embeddings in Vector DB
###Retrieval function (RAG)
### most relevant recipe asked by user
def retrieve_context(query, top_k = 3):
  query_embeddings = model.encode([query])
  distances,indices = index.search(np.array(query_embeddings),top_k)
  results = [chunks[i] for i in indices[0]]
  return "\n".join(results)


In [ ]:
query = "How do i make Vegetable Fried Rice"
retrieve_context(query,top_k = 3)

In [ ]:
def generate_answer(query):
  context = retrieve_context(query, top_k = 3 )

  system_prompt = "You are a expert chef, please answer the query in simple manner."

  user_prompt = f"""
  Answer the following user query - {query} based on the context
  shared {context} to get the recipes of the food for prepartion.
  Use bullet points to answer the query.
  """
  response = client.models.generate_content(
    model ="gemini-2.5-flash",
    contents = f"{system_prompt}\n\n{user_prompt}"
      )
    ##print(response.text)
  return response.text



In [ ]:
while True:
    question = input("\nEnter your recipe questions (type exit to stop): ")

    if question.lower() == "exit":
        print("Thank you for using the BOT")
        break

    answer = generate_answer(question)
    print("\nBot:", answer)

In [ ]:
###Food_Recipes.txt....>Chunking....>Embeddings........>Faiss Vector DB to store data.......>User question.....>Retrieve relevant recipes(context).....>Context+
query......>By Gemini LLM..........>Final Answer

In [ ]:
import pandas as pd
df = {'country': ['USA','China','India','Japan','Mexico'],
      'population':[140,1400,1000,300,250],
      'GDP':[3.73,25.42,8.9,7.9,29],
      'Revenue':[2300,5400,7800,2300,4500]}
df = pd.DataFrame(df)
print(df)


In [ ]:
table_text = df.to_markdown(index=True)
table_text

In [ ]:
### pass the data to LLM............>LLm will give answer

In [ ]:
def answer(query)
  response = client.models.generate_content(models ="gemini-2.5-flash", contents=f"{syatem_prompt}\n\n{user_prompt}")
  return response.text

question = input("\nEnter your question (type exit to stop): ")
if question.lower() == "exit":
    print("Thank you for using the BOT")

    system_prompt = "You are a Data Analyst to answer user query"

    user_prompt = f"""
    Use the following table {table_text} to answer the user query-
    {question}
    """
    answer =answer(question)
    print("\nBot:", answer)

